In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import joblib

# Load trained ML models
rf_model  = joblib.load("random_forest_model.pkl")
xgb_model = joblib.load("xgboost_model.pkl")
iso_model = joblib.load("isolation_forest_model.pkl")  # NEW
scaler = joblib.load("scaler.pkl")
feature_columns = joblib.load("feature_columns.pkl")
print("Random Forest, XGBoost & Isolation Forest models loaded")

Random Forest, XGBoost & Isolation Forest models loaded


In [2]:
def load_sysmon(xml_path):
    with open(xml_path, "r", encoding="utf-8-sig") as f:  # utf-8-sig removes BOM
        content = f.read()

    # Remove namespace
    content = content.replace(
        'xmlns="http://schemas.microsoft.com/win/2004/08/events/event"', ""
    )

    # Remove any XML declaration like <?xml version="1.0"?>
    import re
    content = re.sub(r'<\?xml[^>]+\?>', '', content).strip()

    # Wrap if no root
    if not content.strip().startswith("<Events>"):
        content = "<Events>" + content + "</Events>"

    root = ET.fromstring(content)

    event_count     = {}
    thread_counts   = []
    handle_counts   = []
    dll_names       = set()
    injection_count = 0
    service_count   = 0

    for event in root.findall(".//Event"):
        eid_el = event.find(".//EventID")
        if eid_el is None:
            continue

        eid = int(eid_el.text)
        event_count[eid] = event_count.get(eid, 0) + 1

        data = {}
        for item in event.findall(".//Data"):
            name  = item.get("Name")
            value = item.text
            if name and value:
                data[name] = value

        if eid == 1:
            threads = data.get("NumberOfThreads", "0")
            try:
                thread_counts.append(int(threads))
            except:
                thread_counts.append(0)

        if eid == 7:
            dll = data.get("ImageLoaded", "")
            if dll:
                dll_names.add(dll.lower())

        if eid == 8:
            injection_count += 1

        if eid == 10:
            handle_counts.append(1)

        if eid == 13:
            service_count += 1

    event_count["avg_threads"] = (sum(thread_counts) / len(thread_counts)) if thread_counts else 0
    event_count["avg_handlers"] = len(handle_counts) / max(event_count.get(1, 1), 1)
    event_count["dll_count_real"] = len(dll_names)
    event_count["injection_count"] = injection_count
    event_count["service_count"] = service_count

    return event_count

In [3]:
def extract_features(event_count):

    process_count        = event_count.get(1, 0)   # Process Create
    dll_events           = event_count.get(7, 0)   # Image Load
    handle_events        = event_count.get(10, 0)  # Process Access
    network_events       = event_count.get(3, 0)   # Network Connect
    file_events          = event_count.get(11, 0)  # File Create
    registry_events      = event_count.get(13, 0)  # Registry

    # Real extracted values
    avg_threads   = event_count.get("avg_threads", 0)
    avg_handlers  = event_count.get("avg_handlers", 0)
    dll_count     = event_count.get("dll_count_real", dll_events)
    injections    = event_count.get("injection_count", 0)
    service_count = event_count.get("service_count", 0)

    total = sum(v for k, v in event_count.items() if isinstance(k, int))

    #  UPDATED FEATURES (aligned with training)

    features = {

        "process_count":        process_count,
        "parent_process_count": process_count,

        "avg_threads":          avg_threads,
        "avg_handlers":         avg_handlers,

        "dll_count":            dll_count,
        "dlls_per_process":     dll_count / (process_count + 1),

        #  FIXED FEATURES (important)

        "handles_total":        handle_events * 5,   # scaled
        "handles_avg":          avg_handlers / (process_count + 1),

        "service_count":        service_count,

        "process_services":     registry_events * 2,  # approx mapping

        "mal_injections":       injections,

        "mal_commit_charge":    network_events * 1,   # scaled

        "total_activity":       total,

        "process_density":      process_count / (total + 1),

        "handle_density":       (handle_events * 5) / (process_count + 1),
    }

    #  DEBUG (VERY IMPORTANT)
    print("Final Sysmon Features:", features)

    return features

In [4]:
def behavioral_score(f):
    score = 0

    # Process behavior
    if f["process_count"] > 50:
        score += 1
    if f["process_density"] > 0.5:    # too many processes relative to activity
        score += 1

    # Handle/Memory behavior
    if f["handles_total"] > 100:
        score += 1
    if f["handle_density"] > 10:      # too many handles per process
        score += 1

    # Injection behavior (strongest indicator)
    if f["mal_injections"] > 10:
        score += 2                     # double weight - very suspicious
    
    # DLL behavior
    if f["dll_count"] > 100:
        score += 1
    if f["dlls_per_process"] > 5:     # too many DLLs per process
        score += 1

    # Service/Registry behavior
    if f["service_count"] > 50:
        score += 1

    # Network behavior
    if f["mal_commit_charge"] > 20:  # too many network connections
        score += 1

    # Overall activity
    if f["total_activity"] > 50000:
        score += 1

    return score

In [5]:
def ml_prediction(features):
    import pandas as pd

    df = pd.DataFrame([features])

    # Step 1: Add missing columns
    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0

    # Step 2: Fix column order
    df = df[feature_columns]

    #  DEBUG (important for mismatch issues)
    print("Before scaling:", df)

    # Step 3: Apply scaling
    df = scaler.transform(df)
    df = pd.DataFrame(df, columns=feature_columns)

    #  DEBUG
    print("After scaling:", df.shape)

    # Step 4: Predictions
    rf_pred = rf_model.predict(df)[0]
    xgb_pred = xgb_model.predict(df)[0]

    return rf_pred, xgb_pred

In [6]:
# ---- Load XML ----
xml_file = r"C:\Users\bijar\Desktop\normaldemo.xml"

# ---- Load + extract ----
event_count = load_sysmon(xml_file)
features = extract_features(event_count)

total_activity = features["total_activity"]
process_count  = features["process_count"]

total_feature_sum = sum(v for v in features.values() if isinstance(v, (int, float)))

# ---- STEP 1: Invalid / Empty Check ----
if total_feature_sum <= 5:
    decision = "BENIGN"
    print("\n Very few events → BENIGN")

# ---- STEP 2: Minimum Activity Check ----
elif total_activity < 30 and process_count < 5:
    decision = "BENIGN"
    print("\n Low system activity → BENIGN")

# ---- STEP 3: Full Detection ----
else:

    # Behavioral
    b_score = behavioral_score(features)

    # ML
    rf_pred, xgb_pred = ml_prediction(features)

    #  FIXED: Isolation Forest with scaling
    df_features = pd.DataFrame([features])[feature_columns]
    df_features = scaler.transform(df_features)
    iso_pred = iso_model.predict(df_features)[0]

    votes = 0

    # Behavioral
    if b_score >= 4:
        votes += 1
        behavioral_result = "Ransomware"
    else:
        behavioral_result = "Benign"

    # Random Forest
    if rf_pred == 1:
        votes += 1
        rf_result = "Ransomware"
    else:
        rf_result = "Benign"

    # XGBoost
    if xgb_pred == 1:
        votes += 1
        xgb_result = "Ransomware"
    else:
        xgb_result = "Benign"

    # Isolation Forest
    if iso_pred == -1:
        votes += 1
        iso_result = "Anomaly/Zero Day"
    else:
        iso_result = "Normal"

    #  FINAL DECISION (CORRECTED + INDENTED)
    if votes >= 3:
        decision = "RANSOMWARE"

    elif votes == 2:
        # borderline → check activity
        if total_activity > 30:
            decision = "RANSOMWARE"
        else:
            decision = "BENIGN"

    else:
        decision = "BENIGN"

    # ---- Output ----
    print("\n🔍 Detection Results")
    print("=" * 30)
    print("Behavioral Score :", b_score, "→", behavioral_result)
    print("Random Forest    :", rf_result)
    print("XGBoost          :", xgb_result)
    print("Anomaly Detection:", iso_result)
    print("=" * 30)
    print("Total Votes      :", votes, "/ 4")

    if decision == "RANSOMWARE":
        print("\n FINAL DECISION: RANSOMWARE")
    else:
        print("\n FINAL DECISION: BENIGN")

Final Sysmon Features: {'process_count': 3, 'parent_process_count': 3, 'avg_threads': 0.0, 'avg_handlers': 0.3333333333333333, 'dll_count': 0, 'dlls_per_process': 0.0, 'handles_total': 5, 'handles_avg': 0.08333333333333333, 'service_count': 1, 'process_services': 2, 'mal_injections': 0, 'mal_commit_charge': 1, 'total_activity': 9, 'process_density': 0.3, 'handle_density': 1.25}

 Low system activity → BENIGN


In [7]:
import joblib
rf_model = joblib.load("random_forest_model.pkl")
feature_columns = joblib.load("feature_columns.pkl")
print(feature_columns)

['process_count', 'parent_process_count', 'avg_threads', 'avg_handlers', 'dll_count', 'dlls_per_process', 'handles_total', 'handles_avg', 'service_count', 'process_services', 'mal_injections', 'mal_commit_charge', 'total_activity', 'process_density', 'handle_density']


In [8]:
rf_pred, xgb_pred = ml_prediction(features)
print(rf_pred, xgb_pred)

Before scaling:    process_count  parent_process_count  avg_threads  avg_handlers  dll_count  \
0              3                     3          0.0      0.333333          0   

   dlls_per_process  handles_total  handles_avg  service_count  \
0               0.0              5     0.083333              1   

   process_services  mal_injections  mal_commit_charge  total_activity  \
0                 2               0                  1               9   

   process_density  handle_density  
0              0.3            1.25  
After scaling: (1, 15)
1 1
